# Buckaroo + xorq — push-down stat pipeline

`XorqStatPipeline` runs buckaroo's summary-stat pipeline against a
**xorq expression** (xorq's vendored ibis) instead of pulling the
table into pandas first. Stats compile to a single batched
`expr.aggregate(...)` query plus per-column histogram queries —
computation is pushed to the backend (DuckDB, Postgres, Snowflake,
DataFusion, …) via xorq.

Install the optional dep with `pip install 'buckaroo[xorq]'`.

In [ ]:
import xorq.api as xo

from buckaroo.pluggable_analysis_framework.xorq_stat_pipeline import XorqStatPipeline, XorqDfStatsV2
from buckaroo.pluggable_analysis_framework.stat_func import stat, XorqColumn
from buckaroo.customizations.xorq_stats_v2 import XORQ_STATS_V2
import pandas as pd
expr = xo.memtable({
    'price': [12.5, 18.9, 7.4, 22.1, None, 14.0, 9.9, 31.2, 11.5, 19.8],
    'qty': [1, 2, 1, 3, 1, 2, 4, 1, 2, 5],
    'category': ['a', 'b', 'a', 'c', 'a', 'b', 'b', 'c', 'a', 'b'],
    'is_promo': [True, False, False, True, False, False, True, True, False, True],
})

## 1. Build a xorq table

`xo.memtable` accepts a dict of columns directly — no pandas
required at the construction site. The result is a xorq
expression backed by DataFusion. Any other source works the
same way: `xo.read_parquet`, `xo.read_postgres`,
`xo.connect('duckdb://…').table('events')`, etc.

In [ ]:

expr.execute()

In [ ]:
import buckaroo

In [ ]:
buckaroo.buckaroo_widget.BuckarooInfiniteWidget(expr.execute())

In [ ]:
buckaroo.buckaroo_widget.XorqBuckarooInfiniteWidget(expr) 
#we can make it such that this calls to_record_batches



In [ ]:
next(expr.to_pyarrow_batches())

In [ ]:
next(expr.head(50).to_pyarrow_batches())

In [ ]:
expr

## 2. Run the pipeline

`XorqStatPipeline(stat_funcs).process_table(table)` returns
`(summary_dict, errors)`. Each column gets a flat dict of stats —
the same shape that `DfStatsV2.sdf` produces for pandas/polars.

In [ ]:
pipeline = XorqStatPipeline(XORQ_STATS_V2)
summary, errors = pipeline.process_table(expr)
assert errors == []
# summary is a plain dict-of-dicts — no pandas required.
summary['price']

In [ ]:
# Convenience: cross-column summary as a DataFrame for readability.
pd.DataFrame.from_dict(summary, orient='index')[
    ['_type', 'length', 'null_count', 'distinct_count', 'min', 'max', 'mean', 'std']
]

## 3. What query did we actually run?

Every `@stat` function whose only input is an `XorqColumn` returns
an ibis expression — the pipeline folds them all into a single
`expr.aggregate(...)` call. One round-trip to the backend per
table, no matter how many columns or stats.

In [ ]:
# Reconstruct the same batch query the pipeline builds.
agg_exprs = {
    'price__null_count': expr.price.isnull().sum().coalesce(0).cast('int64'),
    'price__min': expr.price.min().cast('float64'),
    'price__max': expr.price.max().cast('float64'),
    'price__mean': expr.price.mean().cast('float64'),
    'qty__null_count': expr.qty.isnull().sum().coalesce(0).cast('int64'),
    'qty__min': expr.qty.min().cast('float64'),
    'qty__max': expr.qty.max().cast('float64'),
    'qty__mean': expr.qty.mean().cast('float64'),
    'category__distinct_count': expr.category.nunique().cast('int64'),
    'category__null_count': expr.category.isnull().sum().coalesce(0).cast('int64'),
}
batch_query = expr.aggregate(**agg_exprs)
print(xo.to_sql(batch_query))

In [ ]:
# Actually run the expression — one round-trip to the backend, one row of scalars.
batch_result = batch_query.execute()
batch_result

## 4. Histograms

Histograms can't be folded into the batch — each column needs its
own `GROUP BY` query. The pipeline runs them in a second phase,
still on the backend (no pulling rows into Python). Numeric
columns get 10 equal-width buckets; non-numeric columns get the
top-10 categories.

In [ ]:
expr.execute()

In [ ]:
summary['price']['histogram']

In [ ]:
summary['category']['histogram']

In [ ]:
# What 'histogram' actually does under the hood for the 'category' column:
# group_by + count + order — built and executed via xorq.
hist_query = (
    expr.group_by('category')
         .aggregate(n=lambda t: t.count())
         .order_by(xo.desc('n'))
         .limit(10)
)
print(xo.to_sql(hist_query))
hist_query.execute()

## 5. DfStats wrapper for DataFlow integration

`XorqDfStatsV2` mirrors the `DfStatsV2` / `PlDfStatsV2` interface
(`.sdf`, `.errs`, `.add_analysis`) so DataFlow,
`CustomizableDataflow`, and any DfStats consumer can plug a xorq
table in unchanged.

In [ ]:
stats = XorqDfStatsV2(expr, XORQ_STATS_V2)
pd.DataFrame.from_dict(stats.sdf, orient='index')[
    ['_type', 'length', 'null_count', 'distinct_count', 'mean', 'std', 'distinct_per']
]

## 6. Three custom stats from scratch — with a dependency

`@stat`-decorated functions come in two flavours:

- **Batched aggregate** — parameter `col: XorqColumn`, returns an ibis
  expression. The pipeline folds it into the single `expr.aggregate(...)`
  call alongside every other batched stat, so it costs nothing extra.
- **Computed** — parameters are scalar stats from *other* `@stat`
  functions (declared via type-annotated argument names that match
  another stat's name). Runs after the batch resolves, in the typed
  DAG order; no SQL emitted.

Below: `q1` and `q3` are batched (one quartile expression each, both
folded into the same backend round-trip); `iqr` is computed and
declares both as inputs, so the DAG runs `q1` and `q3` first and
hands the resolved floats to `iqr`.

In [ ]:
# Two batched stats — quartiles. Each returns an ibis expression that
# the pipeline folds into the single `expr.aggregate(...)` call.
@stat(column_filter=lambda dt: dt.is_numeric() and not dt.is_boolean())
def q1(col: XorqColumn) -> float:
    return col.quantile(0.25).cast('float64')

@stat(column_filter=lambda dt: dt.is_numeric() and not dt.is_boolean())
def q3(col: XorqColumn) -> float:
    return col.quantile(0.75).cast('float64')

# Computed stat — depends on q1 and q3 by parameter name.
# No XorqColumn here; the pipeline resolves q1/q3 first and passes
# their scalar values in. Same column_filter so the dependency
# survives DAG filtering on non-numeric columns.
@stat(column_filter=lambda dt: dt.is_numeric() and not dt.is_boolean())
def iqr(q1: float, q3: float) -> float:
    if q1 is None or q3 is None:
        return None
    return q3 - q1

# Run with just these three — no XORQ_STATS_V2 needed. The pipeline
# pre-populates `dtype`/`length`; everything else is what we defined.
summary, _ = XorqStatPipeline([q1, q3, iqr]).process_table(expr)
summary


In [ ]:
pd.DataFrame.from_dict(summary, orient='index')[['dtype', 'q1', 'q3', 'iqr']]

## 7. The same IQR as a raw xorq expression

The pipeline gives us DAG ordering, column filtering, and a place
for the result in each column's accumulator — but the underlying
work is just an `expr.aggregate(...)` call. Here's the same IQR
computation written by hand for two columns: `q1`, `q3`, and the
difference all live in a single aggregate, one round-trip to the
backend.

Note that `iqr` doesn't need a separate computed phase here —
because we're inlining everything in one expression, the engine
computes `quantile(.75) - quantile(.25)` in SQL. The pipeline's
two-phase design (batch then computed) is what lets you reuse
`q1` and `q3` as inputs to *other* stats without reissuing the
quantile work.

In [ ]:
# Same answer, no decorators — `q1`, `q3`, and `iqr` all live in
# a single aggregate call. One backend round-trip for two columns.
agg = expr.aggregate(
    price_q1=expr.price.quantile(0.25).cast('float64'),
    price_q3=expr.price.quantile(0.75).cast('float64'),
    price_iqr=(expr.price.quantile(0.75) - expr.price.quantile(0.25)).cast('float64'),
    qty_q1=expr.qty.quantile(0.25).cast('float64'),
    qty_q3=expr.qty.quantile(0.75).cast('float64'),
    qty_iqr=(expr.qty.quantile(0.75) - expr.qty.quantile(0.25)).cast('float64'),
)
print(xo.to_sql(agg))
agg.execute()

## 8. Bring your own backend

Pass `backend=` to route every query (batched aggregate **and**
per-column histograms) through a specific xorq backend. Useful
when the table you have is unbound, or when you want to force
execution on a particular engine.

```python
con = xo.connect('duckdb://')          # or postgres, snowflake, ...
table = con.read_parquet('s3://.../events.parquet')
summary, _ = XorqStatPipeline(XORQ_STATS_V2, backend=con).process_table(table)
```

The aggregate and histogram queries run on `con`; only the small
scalar results come back to Python.